# Medallion artifact check
This notebook verifies that the bronze and silver DuckDB artifacts exist, then previews the tables so you can inspect the actual data.

## What this checks
- whether the DuckDB database exists
- whether the bronze and silver schema/table are present
- a few sample rows from each table

In [ ]:
from pathlib import Path

import duckdb

DB_PATH = Path("/workspaces/financial-transactions-etl/data/db/transactions.duckdb")
print(f"DB exists: {DB_PATH.exists()}")
print(f"DB path: {DB_PATH}")

assert DB_PATH.exists(), f"Missing database at {DB_PATH}"

con = duckdb.connect(str(DB_PATH))
print("DuckDB connected")

schema_rows = con.execute(
    "SELECT table_schema, table_name FROM information_schema.tables WHERE table_schema IN ('bronze','silver') ORDER BY table_schema, table_name"
).fetchall()
print("Medallion tables:")
for row in schema_rows:
    print(" -", row[0], row[1])

assert any(r[0] == "bronze" and r[1] == "transaction" for r in schema_rows), (
    "Missing bronze.transaction"
)
assert any(r[0] == "silver" and r[1] == "transaction" for r in schema_rows), (
    "Missing silver.transaction"
)

In [ ]:
bronze_count = con.execute('SELECT COUNT(*) FROM bronze."transaction"').fetchone()[0]
silver_count = con.execute('SELECT COUNT(*) FROM silver."transaction"').fetchone()[0]

print(f"Bronze rows: {bronze_count}")
print(f"Silver rows: {silver_count}")
print("Bronze columns:", con.execute('DESCRIBE bronze."transaction"').fetchall())
print("Silver columns:", con.execute('DESCRIBE silver."transaction"').fetchall())

In [ ]:
bronze_df = con.execute('SELECT * FROM bronze."transaction" LIMIT 10').fetchdf()
bronze_df.head()

In [ ]:
silver_df = con.execute('SELECT * FROM silver."transaction" LIMIT 10').fetchdf()
silver_df.head()